# AYA Edge pilot data quality audit

This notebook reads only the reviewed source registry and generated summary reports by default. It does not print raw text or automatically acquire data. Before opening any raw payloads, verify license approval and use a bounded sample; treat regex privacy checks as incomplete.

In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd

ROOT = Path.cwd()
DATA_ROOT = Path(__import__('os').environ.get('AYA_DATA_ROOT', ROOT / 'data'))
registry = yaml.safe_load((ROOT / 'configs/data/source_registry.yaml').read_text())
sources = pd.DataFrame(registry['sources'])
sources[['source_id', 'dataset_name', 'revision', 'configuration', 'split', 'license_name', 'approval_status', 'use_class']]

## License and revision gate

Keep conditional evaluation-only approvals separate from training approvals. A row marked `blocked_pending_review` must not be enabled in `acquisition.yaml`.

In [ ]:
audit_path = DATA_ROOT / 'manifests/license_audit.jsonl'
if audit_path.exists():
    license_audit = pd.read_json(audit_path, lines=True)
    display(license_audit[['source_id', 'approval_status', 'license_verified', 'gate', 'reason', 'use_class']])
else:
    print('No generated license_audit.jsonl yet; run verify_licenses.py.')

## Quality and reproducibility summaries

The following section reads counts and lengths only, not the underlying text. It is safe to run when a processed pilot exists; absence of a report is reported as not available rather than as a passing result.

In [ ]:
reports = DATA_ROOT / 'reports'
summary = {}
for name in ['normalization', 'deduplication', 'splits', 'token_counts', 'mixture', 'validation']:
    path = reports / f'{name}.json'
    summary[name] = json.loads(path.read_text()) if path.exists() else None
pd.Series({name: ('not available' if value is None else value.get('status', value.get('actual_total_tokens', value.get('final_records', value.get('valid_records', 'available'))))) for name, value in summary.items()}, name='summary')

In [ ]:
# Inspect split counts and text-length statistics without displaying examples.
import json as _json
rows = []
for split in ['train', 'validation', 'test']:
    path = DATA_ROOT / f'processed/pilot/{split}.jsonl'
    if not path.exists():
        continue
    with path.open() as stream:
        for line in stream:
            item = _json.loads(line)
            if isinstance(item.get('text'), str):
                text = item['text']
            elif isinstance(item.get('messages'), list):
                text = ' '.join(str(message.get('content', '')) for message in item['messages'])
            else:
                text = _json.dumps(item.get('events', []), ensure_ascii=False)
            rows.append({'split': split, 'source_id': item.get('source_id'), 'category': item.get('category'), 'chars': len(text), 'words': len(text.split()), 'sample_id': item.get('sample_id', item.get('trajectory_id'))})
quality = pd.DataFrame(rows)
if quality.empty:
    print('No processed samples are available; this is not a quality pass.')
else:
    display(quality.groupby(['split', 'source_id', 'category']).agg(records=('sample_id', 'count'), median_chars=('chars', 'median'), max_chars=('chars', 'max'), median_words=('words', 'median')).reset_index())

In [ ]:
# Compare 50/50 category targets with actual token counts when a mixture exists.
mixture = summary.get('mixture')
if not mixture:
    print('No mixture report exists; token balance is not available.')
else:
    target = mixture.get('targets', {})
    actual = mixture.get('actual', {})
    display(pd.DataFrame([{'category': key, 'target_tokens': target.get(key, 0), 'actual_tokens': actual.get(key, 0), 'deviation': actual.get(key, 0) - target.get(key, 0)} for key in sorted(set(target) | set(actual))]))
    print('tokenizer:', mixture.get('tokenizer_id'), '| production:', mixture.get('production_tokenizer'))

## Interpretation

Do not call a corpus accepted unless license checks, schema/provenance validation, contamination controls, production token counts, mixture targets, and manual sample review all pass. Save summary outputs only; do not print or commit raw samples or rejected records containing personal data.